Reference Paper: [The Link Prediction Problem for Social Networks](https://www.cs.cornell.edu/home/kleinber/link-pred.pdf)

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import scipy.sparse as sp

dataset = ["dataset/soc-redditHyperlinks-body.tsv", "dataset/soc-redditHyperlinks-title.tsv"]
split_ratio = 0.8
kappa = 50
pagerank_alpha = 0.15
katz_beta = 0.005

In [2]:
frames = []
for subset in dataset:
    df = pd.read_csv(subset, sep='\t', usecols=[0, 1, 3],
                    names=["src", "tgt", "time"], header=0)
    frames.append(df)
df = pd.concat(frames, ignore_index=True)
df['time'] = pd.to_datetime(df['time']).astype('int64')
df = df.sort_values("time").reset_index(drop=True)
all_nodes  = pd.unique(df[["src", "tgt"]].values.ravel())

split_ratio = 0.8
split_idx = int(split_ratio * len(df))
train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]

# Create Core set of active nodes which have at least kappa interactions in both train and test sets
train_deg  = (pd.concat([train_df["src"], train_df["tgt"]]).value_counts())
test_deg   = (pd.concat([test_df["src"],  test_df["tgt"]]).value_counts())
core_nodes = set(train_deg[train_deg >= kappa].index) & set(test_deg [test_deg  >= kappa ].index)

# E_old : edges in training period (directed)
e_old = set(zip(train_df["src"], train_df["tgt"]))
# E_new : NEW directed edges between Core nodes in test period
test_core = test_df[test_df["src"].isin(core_nodes) & test_df["tgt"].isin(core_nodes)]
e_new = (set(zip(test_core["src"], test_core["tgt"])) - e_old)
print(f"Nodes: {len(all_nodes)}  |  Core: {len(core_nodes)}")
print(f"E_old (train set): {len(e_old)}  |  E_new (to predict): {len(e_new)}")

Nodes: 67180  |  Core: 1013
E_old (train set): 278824  |  E_new (to predict): 9410


In [3]:
G = nx.Graph()
G.add_edges_from(e_old)

def candidate_pairs(core_nodes, e_old):
    """Candidate set: Core x Core pairs NOT in E_old and not self-loops."""
    core_list = sorted(core_nodes)
    pairs = []
    for u in core_list:
        for v in core_list:
            if u != v and (u, v) not in e_old:
                pairs.append((u, v))
    return pairs

n_predict = len(e_new)
pairs = candidate_pairs(core_nodes, e_old)
print("Total Candidate pairs:", len(pairs))

def evaluate_predictor(scores_dict, e_new, n_predict):
    """
    Given a dict {(u,v): score}, rank pairs in descending order,
    take top-n_predict, return hit-count and factor-over-random.
    n_predict = |E_new|  (paper's evaluation protocol).
    """
    ranked = sorted(scores_dict.items(), key=lambda x: -x[1])
    top_n  = [pair for pair, _ in ranked[:n_predict]]
    hits   = sum(1 for p in top_n if p in e_new)
    # Random baseline: n_predict random picks from |candidates| pairs
    n_cand  = len(scores_dict)
    if n_cand == 0:
        return 0, 0.0
    p_random = n_predict / n_cand          # prob. a random pick is correct
    expected_random = p_random * n_predict
    factor = hits / expected_random if expected_random > 0 else 0.0
    return hits, factor

results = {}

Total Candidate pairs: 970696


- Methods based on node neighborhoods: Common Neighbors, Jaccard's coefficient, Adamic/Adar, Preferential Attachment
- Methods based on the ensemble of paths: GraphDistance, Katz
- Methods based on ranking: RootedPageRank

In [4]:
def score_common_neighbors(G, u, v):
    return len(set(G.neighbors(u)) & set(G.neighbors(v)))

print("Computing Common Neighbors: ", end="", flush=True)
s = {(u, v): score_common_neighbors(G, u, v) for u, v in pairs}
hits, factor = evaluate_predictor(s, e_new, n_predict)
results["Common Neighbors"] = (hits, factor)
print(f"hits={hits}, factor={factor:.2f}x")

Computing Common Neighbors: hits=506, factor=5.55x


In [5]:
def score_jaccard(G, u, v):
    nu, nv = set(G.neighbors(u)), set(G.neighbors(v))
    denom  = len(nu | nv)
    return len(nu & nv) / denom if denom else 0.0

print("Computing Jaccard's Coefficient: ", end="", flush=True)
s = {(u, v): score_jaccard(G, u, v) for u, v in pairs}
hits, factor = evaluate_predictor(s, e_new, n_predict)
results["Jaccard"] = (hits, factor)
print(f"hits={hits}, factor={factor:.2f}x")

Computing Jaccard's Coefficient: hits=497, factor=5.45x


In [6]:
def score_adamic_adar(G, u, v):
    common = set(G.neighbors(u)) & set(G.neighbors(v))
    score  = 0.0
    for z in common:
        deg = G.degree(z)
        if deg > 1:
            score += 1.0 / np.log(deg)
    return score

print("Computing Adamic/Adar: ", end="", flush=True)
s = {(u, v): score_adamic_adar(G, u, v) for u, v in pairs}
hits, factor = evaluate_predictor(s, e_new, n_predict)
results["Adamic/Adar"] = (hits, factor)
print(f"hits={hits}, factor={factor:.2f}x")

Computing Adamic/Adar: hits=512, factor=5.61x


In [7]:

def score_preferential_attachment(G, u, v):
    return G.degree(u) * G.degree(v)

print("Computing Preferential Attachment: ", end="", flush=True)
s = {(u, v): score_preferential_attachment(G, u, v) for u, v in pairs}
hits, factor = evaluate_predictor(s, e_new, n_predict)
results["Pref. Attachment"] = (hits, factor)
print(f"hits={hits}, factor={factor:.2f}x")

Computing Preferential Attachment: hits=441, factor=4.83x


In [8]:
def score_graph_distance(G, u, v):
    """Negated shortest-path length (undirected)."""
    try:
        return -nx.shortest_path_length(G, u, v)
    except nx.NetworkXNoPath:
        return -np.inf

print("Computing Graph Distance: ", end="", flush=True)
s = {(u, v): score_graph_distance(G, u, v) for u, v in pairs}
hits, factor = evaluate_predictor(s, e_new, n_predict)
results["Graph Distance"] = (hits, factor)
print(f"hits={hits}, factor={factor:.2f}x")

Computing Graph Distance: hits=281, factor=3.08x


In [4]:
def build_katz_scores(G, pairs, beta=0.005, weighted=False):
    """
    score(x,y) = (I - β A)^{-1} - I  entry-wise.
    For large graphs use truncated power series instead of full inverse.
    Computes Katz scores only for specified pairs using sparse operations.
    Returns a dictionary {(u, v): score}.
    """
    nodes  = sorted(G.nodes())
    n      = len(nodes)
    idx    = {v: i for i, v in enumerate(nodes)}
    if weighted:
        A = nx.to_scipy_sparse_array(G, nodelist=nodes, weight='weight', format='csr')
    else:
        A = nx.to_scipy_sparse_array(G, nodelist=nodes, weight=None, format='csr')
        A.data[:] = 1.0
    # Truncated series: sum_{l=1}^{L} β^l A^l  (L=5 is sufficient for small β)
    L     = 2 # reducing to avoid memory overflow
    S     = sp.csr_matrix((n, n), dtype=float)
    Apow  = sp.eye(n, format='csr')
    for l in range(1, L + 1):
        Apow  = Apow @ A
        S     = S + (beta ** l) * Apow
    S_lil = S.tolil()
    su = {}
    for u, v in pairs:
        if u in idx and v in idx:
            su[(u, v)] = S_lil[idx[u], idx[v]]
    return su

print(f"Computing Katz β={katz_beta}: ", end="", flush=True)
su = build_katz_scores(G, pairs, beta=katz_beta, weighted=False)
hits, factor = evaluate_predictor(su, e_new, n_predict)
results[f"Katz unweighted β={katz_beta}"] = (hits, factor)
print(f"hits = {hits}, unweighted factor={factor:.2f}x")

Computing Katz β=0.005: hits = 373, unweighted factor=4.09x


In [4]:
def rooted_pagerank_scores(G_undir, source, alpha=0.15, max_iter=10):
    """
    Personalized PageRank from source node.
    Returns dict {node: score}.
    """
    personalization = {n: 0.0 for n in G_undir.nodes()}
    personalization[source] = 1.0
    try:
        pr = nx.pagerank(G_undir, alpha=alpha,
                         personalization=personalization,
                         max_iter=max_iter, tol=1e-6)
    except nx.PowerIterationFailedConvergence:
        pr = {n: 0.0 for n in G_undir.nodes()}
        pr[source] = 1.0
    return pr

print(f"Computing Rooted PageRank α={pagerank_alpha}: ", end="", flush=True)
pr_cache = {}
s = {}
for u, v in pairs:
    if u not in pr_cache:
        pr_cache[u] = rooted_pagerank_scores(G, u, pagerank_alpha)
    s[(u, v)] = pr_cache[u].get(v, 0.0)
hits, factor = evaluate_predictor(s, e_new, n_predict)
results[f"Rooted PageRank α={pagerank_alpha}"] = (hits, factor)
print(f"hits={hits}, factor={factor:.2f}x")

Computing Rooted PageRank α=0.15: hits=317, factor=3.48x
